# Text Classification

## Overview

Text classification assigns a predefined label to a document. It is one of the most widely deployed NLP tasks:

- **Sentiment analysis** — positive / negative / neutral opinion
- **Spam detection** — spam vs. ham
- **Topic categorisation** — news, sports, politics, etc.
- **Intent detection** — customer support routing
- **Toxic content moderation**

In this lab you will build a **sentiment classifier** on the **Cornell Movie Review Dataset** (Pang & Lee, 2002) — 2000 movie reviews labelled as positive or negative. We use the classic pipeline:

```
Raw text → Tokenisation → TF-IDF vectors → Logistic Regression → Evaluation
```

## Learning Objectives

1. Implement the complete **text classification pipeline** end-to-end
2. Understand and apply **TF-IDF** feature extraction for classification
3. Interpret **logistic regression** coefficients as feature importances
4. Evaluate a classifier using **precision, recall, F1-score**, and **confusion matrix**
5. Diagnose **overfitting** by comparing train vs. test performance
6. Understand when to use scikit-learn vs. PyTorch for classification

---

## Dataset: Cornell Movie Reviews

Download the dataset and place in the defined folder:
- Source: http://www.nltk.org/nltk_data/ (package `movie_reviews`)
- Direct zip: https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages/corpora/movie_reviews.zip

The dataset contains 1000 positive and 1000 negative reviews, collected from the IMDB website by Bo Pang and Lillian Lee.

In [3]:
import sklearn
from sklearn.datasets import load_files
moviedir = r'./movie_reviews'

In [7]:
import os

# Dataset download
if not os.path.exists(moviedir):
    import urllib.request, zipfile, io
    url = 'https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages/corpora/movie_reviews.zip'
    print("Downloading dataset...")
    response = urllib.request.urlopen(url)
    with zipfile.ZipFile(io.BytesIO(response.read())) as z:
        z.extractall('./')
    print("Dataset downloaded and extracted.")

# 1. Data Loading and Preparation

`sklearn.datasets.load_files` loads a directory of text files, using subdirectory names as class labels. Here the subdirectories are `pos/` and `neg/`.

> **Key split**: We use an **80/20 train/test split** with `random_state=12` for reproducibility. The model is trained *only* on the training set and evaluated on the held-out test set — this is essential to avoid reporting inflated (optimistic) performance metrics.

### Inspect the raw data

In [8]:
movie = load_files(moviedir, shuffle=True)
len(movie.data)

2000

In [9]:
movie.target_names

['neg', 'pos']

In [10]:
movie.data[0]

b"arnold schwarzenegger has been an icon for action enthusiasts , since the late 80's , but lately his films have been very sloppy and the one-liners are getting worse . \nit's hard seeing arnold as mr . freeze in batman and robin , especially when he says tons of ice jokes , but hey he got 15 million , what's it matter to him ? \nonce again arnold has signed to do another expensive blockbuster , that can't compare with the likes of the terminator series , true lies and even eraser . \nin this so called dark thriller , the devil ( gabriel byrne ) has come upon earth , to impregnate a woman ( robin tunney ) which happens every 1000 years , and basically destroy the world , but apparently god has chosen one man , and that one man is jericho cane ( arnold himself ) . \nwith the help of a trusty sidekick ( kevin pollack ) , they will stop at nothing to let the devil take over the world ! \nparts of this are actually so absurd , that they would fit right in with dogma . \nyes , the film is 

In [13]:
movie.target[0]

np.int64(0)

### Split the data between train and test

In [19]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.model_selection import train_test_split

docs_train, docs_test, y_train, y_test = train_test_split(movie.data, movie.target, 
                                                          test_size = 0.20, random_state = 12)


[nltk_data] Downloading package punkt to /Users/dsemedo/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/dsemedo/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


### Compute Word Dictionaries and TF-IDF Representation

We use **NLTK's tokenizer** instead of scikit-learn's default because it handles contractions (*"can't"*, *"it's"*) and punctuation more gracefully.

The vectorizer parameters:
- `min_df=2`: ignore words that appear in fewer than 2 documents (likely typos or rare OCR errors)
- `max_features=3000`: cap vocabulary at 3000 most frequent words to control dimensionality

The resulting `docs_train_counts` matrix has shape **(1600 documents, 3000 vocabulary terms)**.

**TF-IDF reweighting** is then applied using `TfidfTransformer`. As explained in Lab01a, this downweights ubiquitous words and amplifies the signal of informative, document-specific terms.

In [20]:
movieVzer= CountVectorizer(min_df=2, 
        tokenizer=nltk.word_tokenize, max_features=3000)
docs_train_counts = movieVzer.fit_transform(docs_train)
docs_test_counts = movieVzer.transform(docs_test)
docs_train_counts.shape

(1600, 3000)

### TF-IDF weighting


In [21]:
movieTfmer = TfidfTransformer()
docs_train_tfidf = movieTfmer.fit_transform(docs_train_counts)
docs_test_tfidf = movieTfmer.transform(docs_test_counts)

# 2. Model Training — Logistic Regression

**Logistic Regression** is a linear classifier that models the probability of class $y$ given input features $\mathbf{x}$:

$$P(y=1 \mid \mathbf{x}) = \sigma(\mathbf{w}^\top \mathbf{x} + b) = \frac{1}{1 + e^{-(\mathbf{w}^\top \mathbf{x} + b)}}$$

In the multi-class setting, it uses a softmax function. For our binary task (positive/negative), the model learns **one weight per vocabulary term** — so a 3000-dimensional weight vector.

**Key hyperparameters** (see link below):
- `penalty` — L1 (sparse weights, feature selection) or L2 (dense weights, better generalisation)
- `C` — Inverse regularisation strength. Higher C = less regularisation = risk of overfitting
- `class_weight` — Useful for imbalanced datasets (`class_weight='balanced'`)
- `max_iter` — Increase if the solver does not converge

📖 **Reference**: https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html

In [22]:
from sklearn.linear_model import LogisticRegression
clf = LogisticRegression(random_state=0).fit(docs_train_tfidf, y_train)

The model parameters are given by the coef_ variable:

In [23]:
print(clf.coef_)

[[-0.9750461  -0.02708897 -0.0736249  ...  0.09516821  0.07821451
   0.0214086 ]]


# 3. Evaluation

## Classification Report

The `classification_report` provides four key metrics for each class:

| Metric | Formula | Meaning |
|--------|---------|---------|
| **Precision** | TP / (TP + FP) | Of all predicted positives, how many were actually positive? |
| **Recall** | TP / (TP + FN) | Of all actual positives, how many did we find? |
| **F1-score** | 2·P·R / (P+R) | Harmonic mean of precision and recall |
| **Support** | — | Number of samples with this true label |

**Macro avg** = unweighted mean across classes (treats each class equally).  
**Weighted avg** = weighted by support (appropriate for imbalanced datasets).

📖 **Reference**: https://scikit-learn.org/stable/modules/model_evaluation.html

## Diagnosing Overfitting

Compare train (91%) vs test (78%) accuracy:
- The 13-point gap indicates **moderate overfitting** — the model memorises some training patterns that do not generalise
- Solutions: reduce `max_features`, increase regularisation (`C < 1`), add more training data, or use cross-validation

> The test accuracy of ~78% is actually quite good for a linear model on this task. BERT fine-tuned on the same data achieves ~95%+, but requires orders of magnitude more computation.

In [24]:
import numpy as np
from sklearn.metrics import classification_report
predict_train = clf.predict(docs_train_tfidf)
predicted_test = clf.predict(docs_test_tfidf)

### Training results

In [25]:
target_names = ['neg', 'pos']
print(classification_report(predict_train, y_train, target_names=target_names))

              precision    recall  f1-score   support

         neg       0.90      0.91      0.90       786
         pos       0.91      0.90      0.91       814

    accuracy                           0.91      1600
   macro avg       0.90      0.91      0.90      1600
weighted avg       0.91      0.91      0.91      1600



### Test results

In [26]:
print(classification_report(predicted_test, y_test, target_names=target_names))

              precision    recall  f1-score   support

         neg       0.73      0.81      0.77       186
         pos       0.82      0.74      0.78       214

    accuracy                           0.78       400
   macro avg       0.78      0.78      0.77       400
weighted avg       0.78      0.78      0.78       400



# 4. Exercise — PyTorch Implementation

The scikit-learn pipeline above is production-ready but hides the mathematical details. Implement the same classifier in **PyTorch** to see exactly what is happening at each step.

### Architecture

```
TF-IDF vector (3000-dim) → Linear(3000, 2) → Softmax → class probabilities
```

### Steps

1. Convert the `docs_train_tfidf` and `docs_test_tfidf` sparse matrices to dense PyTorch tensors
2. Implement a training loop: forward pass → cross-entropy loss → backward → optimizer step
3. Report accuracy on the test set

### Hints

```python
import torch
import torch.nn as nn

# Convert sparse matrix to dense tensor
X_train = torch.FloatTensor(docs_train_tfidf.toarray())
y_train_tensor = torch.LongTensor(y_train)

# Model
model = nn.Linear(3000, 2)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

# Training loop (fill in)
for epoch in range(100):
    ...
```

### Extension

Replace the linear layer with a small MLP:
```
Linear(3000, 256) → ReLU → Dropout(0.5) → Linear(256, 2)
```

Does adding a hidden layer improve test accuracy?
